# Extract Routing Trace from Ling-3.0-tiny

**128 experts, top-8, sigmoid + group-limited routing**

- 24 слоя, из них 23 MoE (first_k_dense_replace=1)
- 128 routed experts, top-8, 1 shared expert
- Router: sigmoid + group-limited topk (n_group=8, topk_group=4)
- Хук на BailingMoeV3Gate
- Ожидаемо: 1000 токенов × 23 слоя = 23000 записей
- Выход: /content/ling-routing-trace.jsonl

In [ ]:
# Install dependencies
!pip install -q compressed-tensors accelerate bitsandbytes transformers

In [ ]:
# ============================================================
# extract_ling_trace.py — Colab
# ============================================================

import json
import math
import torch
from collections import Counter
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig

import transformers
print(f"transformers version: {transformers.__version__}")

MODEL_ID = "inclusionAI/Ling-3.0-tiny"

print(f"\nLoading config for {MODEL_ID}...")
config = AutoConfig.from_pretrained(MODEL_ID, trust_remote_code=True)

print(f"  architectures:           {config.architectures}")
print(f"  model_type:              {config.model_type}")
print(f"  num_hidden_layers:       {config.num_hidden_layers}")
print(f"  first_k_dense_replace:   {config.first_k_dense_replace}")
print(f"  num_experts:             {config.num_experts}")
print(f"  num_experts_per_tok:     {config.num_experts_per_tok}")
print(f"  num_shared_experts:      {config.num_shared_experts}")
print(f"  n_group:                 {config.n_group}")
print(f"  topk_group:              {config.topk_group}")
print(f"  score_function:          {config.score_function}")
print(f"  topk_method:             {config.topk_method}")
print(f"  output_router_logits:    {config.output_router_logits}")

assert config.num_experts == 128, f"expected 128 experts, got {config.num_experts}"
assert config.num_experts_per_tok == 8, f"expected top-8, got {config.num_experts_per_tok}"

n_moe_layers = config.num_hidden_layers - config.first_k_dense_replace
print(f"\n  MoE layers: {config.first_k_dense_replace}..{config.num_hidden_layers - 1} "
      f"({n_moe_layers} layers)")

print(f"\nLoading model {MODEL_ID}...")
model = None

try:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID + "-int4",
        trust_remote_code=True,
        device_map="auto",
        torch_dtype="auto",
    )
    print("Loaded INT4 checkpoint (compressed-tensors).")
except Exception as e:
    print(f"INT4 load failed: {e}")
    print("Falling back to BF16 + bitsandbytes 4-bit...")

    from transformers import BitsAndBytesConfig
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type="nf4",
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        trust_remote_code=True,
        device_map="auto",
        quantization_config=bnb_config,
    )
    print("Loaded BF16 with on-the-fly 4-bit quantization.")

model.eval()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

gate_class_name = "BailingMoeV3Gate"
hooks = []
trace = []

def make_hook(layer_idx):
    def hook(module, args, output):
        if isinstance(output, tuple) and len(output) >= 2:
            topk_idx = output[0]
            topk_weight = output[1]
        else:
            return

        if topk_idx.dim() == 3:
            topk_idx = topk_idx.reshape(-1, topk_idx.shape[-1])
            topk_weight = topk_weight.reshape(-1, topk_weight.shape[-1])

        n_tokens = topk_idx.shape[0]

        for pos in range(n_tokens):
            experts = sorted(topk_idx[pos].tolist())
            weights = topk_weight[pos].tolist()
            order = sorted(range(len(experts)), key=lambda k: topk_idx[pos][k].item())
            weights_sorted = [weights[i] for i in order]
            trace.append({
                "layer": layer_idx,
                "pos": pos,
                "experts": experts,
                "weights": weights_sorted,
            })

    return hook

for name, module in model.named_modules():
    if module.__class__.__name__ == gate_class_name:
        parts = name.split(".")
        layer_idx = None
        for i, p in enumerate(parts):
            if p == "layers" and i + 1 < len(parts):
                try:
                    layer_idx = int(parts[i + 1])
                except ValueError:
                    pass
                break
        if layer_idx is None:
            continue
        h = module.register_forward_hook(make_hook(layer_idx))
        hooks.append(h)
        print(f"Hooked layer {layer_idx}: {name}")

print(f"\nTotal hooks: {len(hooks)}")
assert len(hooks) == n_moe_layers, f"expected {n_moe_layers} hooks, got {len(hooks)}"

seed_text = (
    "The quick brown fox jumps over the lazy dog. "
    "Machine learning models process text token by token. "
    "Mixture of experts routes each token to specialized subnetworks. "
    "Storage layouts affect memory access patterns. "
    "Cold reads hit the disk, warm reads hit the cache. "
) * 20

inputs = tokenizer(
    seed_text,
    return_tensors="pt",
    truncation=True,
    max_length=1000,
).to(model.device)

n_input = inputs["input_ids"].shape[1]
print(f"\nInput sequence length: {n_input}")

print("Running single forward pass...")
with torch.no_grad():
    _ = model(
        **inputs,
        output_router_logits=True,
        use_cache=False,
    )

for h in hooks:
    h.remove()
print("Hooks removed.")
print(f"Trace size: {len(trace)} records")
print(f"Expected:   {n_input} tokens x {n_moe_layers} layers = {n_input * n_moe_layers}")

OUT_PATH = "/content/ling-routing-trace.jsonl"
trace.sort(key=lambda r: (r["layer"], r["pos"]))

with open(OUT_PATH, "w") as f:
    for rec in trace:
        f.write(json.dumps(rec) + "\n")

print(f"\nWrote {len(trace)} records to {OUT_PATH}")

print("\n" + "=" * 60)
print("ANALYSIS")
print("=" * 60)

per_layer = {}
for r in trace:
    per_layer.setdefault(r["layer"], Counter()).update(r["experts"])

print(f"\nTotal records: {len(trace)}")
print(f"MoE layers present: {sorted(per_layer.keys())}")

print("\n--- Expert frequency and entropy ---")
for layer in sorted(per_layer):
    c = per_layer[layer]
    total = sum(c.values())
    entropy = -sum((v / total) * math.log(v / total) for v in c.values() if v > 0)
    max_entropy = math.log(128)
    top = c.most_common(5)
    print(f"layer {layer}: top5={top}")
    print(f"  entropy = {entropy:.3f} / {max_entropy:.3f} "
          f"({entropy / max_entropy * 100:.1f}%)")

print("\n--- p_adjacent ---")
for layer in sorted(per_layer):
    recs = sorted([r for r in trace if r["layer"] == layer], key=lambda x: x["pos"])
    adj = 0
    for i in range(1, len(recs)):
        if set(recs[i - 1]["experts"]) & set(recs[i]["experts"]):
            adj += 1
    print(f"layer {layer}: p_adjacent = {adj / max(1, len(recs) - 1):.3f}")

print("\n--- p_window_10 ---")
for layer in sorted(per_layer):
    recs = sorted([r for r in trace if r["layer"] == layer], key=lambda x: x["pos"])
    window = 10
    total = 0
    reuse = 0
    for i in range(len(recs)):
        lo = max(0, i - window)
        recent = set()
        for j in range(lo, i):
            recent.update(recs[j]["experts"])
        curr = set(recs[i]["experts"])
        total += 1
        if recent & curr:
            reuse += 1
    print(f"layer {layer}: p_window_{window} = {reuse / total:.3f}")

print("\nDone.")

try:
    from google.colab import files
    files.download(OUT_PATH)
except ImportError:
    print(f"(not in Colab; file at {OUT_PATH})")